# Code pour faire marcher PySpark chez moi 

In [2]:
import os, subprocess, shutil
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["PATH"] = r"C:\hadoop\bin;" + os.environ.get("PATH", "")
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
print("winutils:", os.path.exists(r"C:\hadoop\bin\winutils.exe"))
print("hadoop.dll:", os.path.exists(r"C:\hadoop\bin\hadoop.dll"))

os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot" 
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + ";" + os.environ.get("PATH", "")
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("java in PATH:", shutil.which("java"))
print(subprocess.run(["java","-version"], capture_output=True, text=True).stderr)

winutils: True
hadoop.dll: True
JAVA_HOME: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot
java in PATH: C:\Program Files\Eclipse Adoptium\jdk-17.0.18.8-hotspot\bin\java.EXE
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment Temurin-17.0.18+8 (build 17.0.18+8)
OpenJDK 64-Bit Server VM Temurin-17.0.18+8 (build 17.0.18+8, mixed mode, sharing)



In [3]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

import pyspark.pandas as ps
ps.set_option("compute.fail_on_ansi_mode", False)

print("ansi =", spark.conf.get("spark.sql.ansi.enabled"))

ansi = false


# Code principal tests

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyspark.pandas as ps
from pyspark.sql.functions import *

In [4]:
pers_paris = pd.read_csv(".\data\BDD_BGES\BDD_BGES_PARIS\PERSONNEL_PARIS.txt", sep=";", index_col="ID_PERSONNEL")

In [5]:
pers_paris.head()

,NOM_PERSONNEL,PRENOM_PERSONNEL,DT_NAISS,VILLE_NAISS,PAYS_NAISS,NUM_SECU,IND_PAYS_NUM_TELP,NUM_TELEPHONE,NUM_VOIE,DSC_VOIE,CMPL_VOIE,CD_POSTAL,VILLE,PAYS,FONCTION_PERSONNEL,TS_CREATION_PERSONNEL,TS_MAJ_PPERSONNEL
ID_PERSONNEL,,,,,,,,,,,,,,,,,
KeyPers_Paris_1230000,Name0,FistName0,1992-03-13,Lille,France,NS000000000,NaN,+336##0151713,22,NomVoie100,NaN,#9423,Paris,France,Ingénieur Informaticien,2005-06-24 06:48:21,2005-06-24 06:48:21
KeyPers_Paris_1230001,Name1,FistName1,1955-02-17,Pekin,China,NS000000001,NaN,+336##0797149,33,NomVoie731,NaN,#2429,Paris,France,Ingénieur Data,2007-11-16 22:38:43,2007-11-16 22:38:43
KeyPers_Paris_1230002,Name2,FistName2,1982-10-06,Alger,Algeria,NS000000002,NaN,+336##0319378,55,NomVoie622,NaN,#7861,Paris,France,Ingénieur Informaticien,2004-07-09 14:25:16,2004-07-09 14:25:16
KeyPers_Paris_1230003,Name3,FistName3,1955-08-22,Lima,Peru,NS000000003,NaN,+336##0205027,61,NomVoie363,NaN,#4028,Paris,France,Ingénieur Data,2013-02-16 23:23:05,2013-02-16 23:23:05
KeyPers_Paris_1230004,Name4,FistName4,2017-04-07,Pekin,China,NS000000004,NaN,+336##0456031,68,NomVoie914,NaN,#0111,Paris,France,Cadre,2001-10-13 22:42:51,2001-10-13 22:42:51


Fonction readFiles : réunir tous les .txt d'un dossier en un seul dataframe

In [8]:
def readFiles(path, indexCol):
    p = Path(path)
    files = sorted(p.glob("*.txt"))  #liste des fichiers dans le dossier
    dfs = []
    """
    for f in files:
        df = pd.read_csv(f, sep=";", index_col=indexCol)  
        df["source_file"] = f.name # optionnel si on veut garder le fichier d'où vient l'information              
        dfs.append(df)
    """

    """
    # Sinon en compréhension de liste sans la colonne source_file
    dfs = [pd.read_csv(f, sep=';', index_col=indexCol) for f in files]
    all_df = pd.concat(dfs, verify_integrity=True)  # ValueError si doublons d'index
    """

    # Avec pyspark (très lent)
    dfs = [ps.read_csv(str(f), sep=';', index_col=indexCol) for f in files]
    all_df = ps.concat(dfs) 
    tmp = all_df.reset_index()
    dups = tmp[tmp.duplicated(subset=indexCol, keep=False)][indexCol].unique().to_list()
    if dups:
        raise ValueError(f"Doublons d'{indexCol} détectés: {dups}")
    

    return all_df

In [ ]:
mat_info_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS/BDD_BGES_PARIS_INFORMATIQUE")
mat_info_paris_psdf.shape[0]

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


1545

In [ ]:
mat_info_paris_sdf = mat_info_paris_psdf.to_spark(index_col="ID_MATERIELINFO")

In [21]:
mat_info_paris_sdf.show(10)

+--------------------+--------------------+-------------+----------------+-------------------+------------------+--------------------+
|     ID_MATERIELINFO|        ID_PERSONNEL|NOM_PERSONNEL|PRENOM_PERSONNEL|         DATE_ACHAT|              TYPE|              MODELE|
+--------------------+--------------------+-------------+----------------+-------------------+------------------+--------------------+
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name4777|    FistName4777|2024-04-29 15:43:53|       PC portable|       Latitude 5xxx|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name2309|    FistName2309|2024-04-29 08:23:31|        imprimante|Laser à poser (<4...|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name4981|    FistName4981|2024-04-29 16:04:32|                  |            Mac mini|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name1199|    FistName1199|2024-04-29 16:58:34|      Telephone IP|   modèle par défaut|
|Paris_MATERIEL_IN...|KeyPers_Paris_123...|     Name185

#### Test de réponse à une question 

Combien d’ingénieurs informaticiens travaillent sur le site de Paris ?

In [12]:
pers_paris_psdf = readFiles("data/BDD_BGES/BDD_BGES_PARIS", "ID_PERSONNEL")
pers_paris_sdf = pers_paris_psdf.to_spark(index_col="ID_PERSONNEL")

c:\Users\etien\miniconda3\envs\nf26\Lib\site-packages\pyspark\pandas\utils.py:1037: PandasAPIOnSparkAdviceWarning: `to_list` loads all data into the driver's memory. It should only be used if the resulting list is expected to be small.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


In [19]:
nb_inge_info = (
    pers_paris_sdf
    .filter(col("FONCTION_PERSONNEL") == "Ingénieur Informaticien")
    .count()
)

print(f"{nb_inge_info} ingénieurs informaticiens travaillent sur le site de Paris")

1831 ingénieurs informaticiens travaillent sur le site de Paris
